# Stack NPB Epoch 100 → Full Reconstructed Pair Blocks

Stacks per-channel NPB snapshots (`{protein}_channel_{0..127}.npy`) from `npb_epoch_100/` into full `(L, L, 128)` reconstructed pair blocks.

Output: `sae_reconstructions_epoch_100/{protein}_reconstructed_pair_block_47.npy`

In [ ]:
# Configuration
import os
from pathlib import Path

BASE = os.environ.get("BASE", "/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins")
NPB_DIR = os.path.join(BASE, "sae_results", "layer_47", "npb_epoch_100")
OUTPUT_DIR = os.path.join(BASE, "sae_reconstructions_epoch_100")
NUM_CHANNELS = 128

if not os.path.isdir(NPB_DIR):
    raise FileNotFoundError(f"NPB dir not found: {NPB_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"NPB dir:   {NPB_DIR}")
print(f"Output:    {OUTPUT_DIR}")

In [ ]:
# Discover proteins with all 128 channels
import numpy as np

files = os.listdir(NPB_DIR)
protein_channels = {}  # protein_id -> set of channel indices
for f in files:
    if f.endswith(".npy") and "_channel_" in f:
        # e.g. 6tf4_A_channel_0.npy
        rest = f.replace("_channel_", " ")
        parts = rest.rsplit(".", 1)[0].rsplit(" ", 1)
        if len(parts) == 2:
            protein_id, ch_str = parts
            try:
                ch = int(ch_str)
                protein_channels.setdefault(protein_id, set()).add(ch)
            except ValueError:
                pass

complete = [p for p, chs in protein_channels.items() if len(chs) == NUM_CHANNELS]
complete.sort()
print(f"Proteins with all {NUM_CHANNELS} channels: {len(complete)}")
if not complete:
    raise SystemExit("No complete proteins found.")

In [ ]:
# Stack and save
ok = 0
for protein_id in complete:
    channels = []
    for ch in range(NUM_CHANNELS):
        path = os.path.join(NPB_DIR, f"{protein_id}_channel_{ch}.npy")
        arr = np.load(path)
        channels.append(arr)
    stacked = np.stack(channels, axis=2)  # (L, L, 128)
    out_path = os.path.join(OUTPUT_DIR, f"{protein_id}_reconstructed_pair_block_47.npy")
    np.save(out_path, stacked)
    ok += 1
    if ok <= 3 or ok % 20 == 0:
        print(f"  {protein_id}: {stacked.shape} -> {out_path}")

print(f"\nDone: {ok} proteins stacked to {OUTPUT_DIR}")